# Importation et définition de variables globales

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import random
import ale_py
import gc
import seaborn as sns
import pickle

from agent.rainbow_agent import RainbowAgent

from agent.no_noisy_no_categ_agent import NoNoisyNoCategAgent
from reseaux.nn_no_noisy_no_categ_no_duel import Network as NnNoThree
from reseaux.nn_no_noisy_no_categ import Network as NnDueling

from agent.no_noisy_agent import NoNoisyAgent 
from reseaux.nn_no_noisy_no_duel import Network as NnCateg
from reseaux.nn_no_noisy import Network as NnNoNoisy

from utils.processing import show_latest_video

In [ ]:
# Environment
env = gym.make("ALE/Freeway-v5", render_mode="rgb_array")

# Set random seed

In [ ]:
seed = 777

def seed_torch(seed):
    torch.manual_seed(seed)
    if torch.backends.cudnn.enabled:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

np.random.seed(seed)
random.seed(seed)
seed_torch(seed)

# Initialisation

In [ ]:
# paramètres
num_frames = 100000
ss_num_frames = 5000
memory_size = 100000 #ou 10 000 d'après le code original
batch_size = 128
target_update = 100
epsilon_decay = 1 / 2000
buffer_dir = './buffer_saves'

# Agent

GERER L'ASPECT VIDEO !!!!! ; désactiver, écrasement

In [ ]:
#ordre : rainbow, no_three, no_noisy_no_categ, no_noisy_no_duel, no_noisy

score_agents = np.zeros([1, num_frames//ss_num_frames], dtype=float)
nb_crashes_agents = np.zeros([1, num_frames//ss_num_frames], dtype=int)
# Définition
agent = RainbowAgent(env, memory_size, batch_size, target_update, seed, seed, f'{buffer_dir}/rainbow')
nom_agent = "rainbow"
agent = NoNoisyNoCategAgent(NnNoThree, env, memory_size, batch_size, target_update, epsilon_decay, seed, seed, f'{buffer_dir}/no_three')
#nom_agent = "no_three"
agent = NoNoisyNoCategAgent(NnDueling, env, memory_size, batch_size, target_update, epsilon_decay, seed, seed, f'{buffer_dir}/no_noisy_no_categ')
#nom_agent = "no_noisy_no_categ"
agent = NoNoisyAgent(NnCateg, env, memory_size, batch_size, target_update,  epsilon_decay, seed, seed, f'{buffer_dir}/no_noisy_no_duel')
#nom_agent = "no_noisy_no_duel"
agent = NoNoisyAgent(NnNoNoisy, env, memory_size, batch_size, target_update, epsilon_decay, seed, seed, f'{buffer_dir}/no_noisy')
#nom_agent = "no_noisy"


for iter in range(0, num_frames, ss_num_frames): 
   
   # Entraînement
   agent.train(iter)

   # Test
   score_agents[0][iter//ss_num_frames], nb_crashes_agents[0][iter//ss_num_frames] = agent.test("corbeille/" + nom_agent + "/")

    #Sauvegarde préventive en cas d'interruption
   if iter % ss_num_frames == 0:
      # Sauvegarde
      with open("score_agents_" + nom_agent + "_" + str(num_frames) + ".pkl", "wb") as f:
         pickle.dump(score_agents, f)
      with open("nb_crashes_" + nom_agent + "_" + str(num_frames) + ".pkl", "wb") as g:
         pickle.dump(nb_crashes_agents, g)
      print(f"Liste sauvegardée : {iter}")

In [ ]:
# Exécutez cette cellule, afin de supprimer les fichiers de buffer_save à la main :
agent.cleanup()
del agent

In [ ]:
# Charger le fichier localement
with open("score_agents_" + nom_agent + "_" + str(num_frames) + ".pkl", "rb") as f:
    score_agents = pickle.load(f)
         
with open("nb_crashes_" + nom_agent + "_" + str(num_frames) + ".pkl", "rb") as g:
    nb_crashes_agents = pickle.load(g)

print("Liste chargée :", type(score_agents))  # Vérifie le type de l'objet
print("Liste chargée :", type(nb_crashes_agents))  # Vérifie le type de l'objet

In [ ]:
liste_frames = np.arange(0, num_frames, ss_num_frames)
#liste_label = ["rainbow", "no noisy, no categorical, no dueling", "no noisy, no categorical", "no noisy, no dueling", "no noisy"]
liste_label = [nom_agent]
# Générer une palette de 5 couleurs harmonieuses
colors = sns.color_palette("deep", n_colors=len(liste_label))


# Création de la figure et du graphe
plt.figure(figsize=(8, 6))
plt.title("Graphe avec " + str(len(liste_label)) + " courbes")

# Tracer les courbes
for i in range(len(liste_label)):
    plt.plot(liste_frames, score_agents[i], label=liste_label[i], color=colors[i], linewidth=2)

# Ajout de la légende et des labels
plt.xlabel("Num_frames d'entraînement") # QU EST CE QUE NUM FRAMES DANS TRAI?
plt.ylabel("Score obtenu par la fonction reward pour le test")
plt.legend()

# Affichage du graphe
plt.show()

In [ ]:
liste_frames = np.arange(0, num_frames, ss_num_frames)
#liste_label = ["rainbow", "no noisy, no categorical, no dueling", "no noisy, no categorical", "no noisy, no dueling", "no noisy"]
liste_label = [nom_agent]
# Générer une palette de 5 couleurs harmonieuses
colors = sns.color_palette("deep", n_colors=len(liste_label))


# Création de la figure et du graphe
plt.figure(figsize=(8, 6))
plt.title("Graphe avec " + str(len(liste_label)) + " courbes")

# Tracer les courbes
for i in range(len(liste_label)):
    plt.plot(liste_frames, nb_crashes_agents[i], label=liste_label[i], color=colors[i], linewidth=2)

# Ajout de la légende et des labels
plt.xlabel("Num_frames d'entraînement") # QU EST CE QUE NUM FRAMES DANS TRAI?
plt.ylabel("Score obtenu par la fonction reward pour le test")
plt.legend()

# Affichage du graphe
plt.show()

In [ ]:
score_agents[0]

In [ ]:
nb_crashes_agents[0]